# Reconstruction Error Comparison: Wasserstein SAE vs NMF

This notebook compares two methods for representing hyperspectral data:

1. **Wasserstein SAE**: Sparse autoencoder with monotone atoms (transport maps).  
   Data is pre-embedded as OT maps from `Uniform[0,1]` to the normalized spectrum.
2. **NMF**: Non-negative matrix factorization on normalized spectra.

Two comparisons:
- **Comparison A** — L2 error on reconstructed spectral densities (both methods → spectra)
- **Comparison B** — L2 error on transport map encodings (both methods → OT maps)

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'SAE.py').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
HSI_PIPELINE_DIR = REPO_ROOT / 'hsi' / 'pipeline'
if str(HSI_PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(HSI_PIPELINE_DIR))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.decomposition import NMF
from itertools import product


from SAE import *
from SAE_analysis_functions import ARCH_REGISTRY, sae_encode_potential_batch

## 0. Dataset Configuration

Change this cell to switch datasets. Everything downstream uses these variables.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DATASET CONFIG — edit this block to swap datasets
# ═══════════════════════════════════════════════════════════════
DATASET_NAME   = 'Pavia'
DATA_DIR       = REPO_ROOT / Path('datasets/hsi_data/pavia')
TRANSPORT_MAP_PT = DATA_DIR / 'transport_maps' / 'pavia_cube.pt'   # precomputed OT maps
ORIGINAL_PT    = DATA_DIR / 'data' / 'pavia_cube.pt'         # original spectra
SAE_ROOT       = DATA_DIR / 'SAE_params' / 'transport_maps' / 'unreg' / 'seed_0'
N_BANDS        = 102   # number of spectral bands (= grid size for uniform source)
DEVICE         = 'cpu'

# SAE models to compare (keys must match subdirectories under SAE_ROOT)
SAE_CONFIGS = {
    'JUMPRELUAE_10_1e-2_mon': {'architecture': 'JumpReLUAE_monotone', 'hidden_dim': 10, 'top_K': 0},
    'JUMPRELUAE_10_5e-1_mon': {'architecture': 'JumpReLUAE_monotone', 'hidden_dim': 10, 'top_K': 0},
    'JUMPRELUAE_10_1e-8_mon': {'architecture': 'JumpReLUAE_monotone', 'hidden_dim': 10, 'top_K': 0},
}

# NMF ranks to compare
NMF_RANKS = [10]

# ═══════════════════════════════════════════════════════════════

## 1. Load Data

In [ ]:
# Load precomputed OT maps (Uniform → spectrum) and original spectra
transport_map_raw = torch.load(TRANSPORT_MAP_PT)
transport_map_cube = transport_map_raw['cube'] if isinstance(transport_map_raw, dict) and 'cube' in transport_map_raw else transport_map_raw

og_raw = torch.load(ORIGINAL_PT)
og_cube = og_raw['cube'] if isinstance(og_raw, dict) and 'cube' in og_raw else og_raw

# Flatten spatial dims if needed: (H, W, D) → (N, D)  or keep (N, D)
if transport_map_cube.ndim == 3:
    H, W, D = transport_map_cube.shape
    transport_map_data = transport_map_cube.reshape(H * W, D)
    og_data = og_cube.reshape(H * W, D)
else:
    transport_map_data = transport_map_cube
    og_data = og_cube

N, D = transport_map_data.shape
assert D == N_BANDS, f'Expected {N_BANDS} bands, got {D}'

# Normalize spectra to probability measures (non-negative, sum to 1)
og_np = og_data.numpy() if torch.is_tensor(og_data) else np.asarray(og_data)
og_np = np.clip(og_np, 0, None)
row_sums = og_np.sum(axis=1, keepdims=True)
row_sums = np.where(row_sums < 1e-12, 1.0, row_sums)
spectra_normalized = og_np / row_sums

print(f'Dataset: {DATASET_NAME}')
print(f'N = {N} pixels, D = {D} bands')
print(f'Transport map data shape: {transport_map_data.shape}')
print(f'Spectra shape: {spectra_normalized.shape}')

## 2. Helper Functions

In [ ]:
# ── OT map helpers ──

def normalize_inv_cdf(T, eps=1e-12):
    """Shift to start at 0, scale to end at 1, enforce monotonicity."""
    T = np.asarray(T, dtype=np.float64)
    T = T - T[0]
    denom = T[-1] if abs(T[-1]) > eps else eps
    T = T / denom
    return np.maximum.accumulate(T)


def kde_density_from_inv_cdf(inv_cdf, n_samples=20000, bandwidth=0.05, n_grid=None):
    """
    Convert an inverse-CDF (transport map) to a density via KDE.
    
    1. Sample from Uniform[0,1], push through the inverse-CDF.
    2. KDE the resulting samples.
    3. Evaluate on a grid matching the number of spectral bands.
    
    Returns (x_grid, density) where density integrates to 1.
    """
    inv_cdf = np.asarray(inv_cdf, dtype=np.float64)
    m = len(inv_cdf)
    if n_grid is None:
        n_grid = m
    
    # Normalize to a valid inverse-CDF
    T = normalize_inv_cdf(inv_cdf)
    
    # Sample from Uniform, push through T
    p_grid = np.linspace(0.0, 1.0, m)
    u = np.random.uniform(0.0, 1.0, n_samples)
    samples = np.interp(u, p_grid, T)
    
    # KDE
    x_grid = np.linspace(0.0, 1.0, n_grid)
    kde = stats.gaussian_kde(samples)
    kde.set_bandwidth(bandwidth)
    density = kde(x_grid)
    
    # Normalize to sum to 1 (PMF-style, for L2 comparison with normalized spectra)
    total = density.sum()
    if total > 0:
        density = density / total
    
    return x_grid, density


def spectra_to_ot_maps(spectra, n_bands):
    """
    Convert normalized spectra (probability measures on a uniform grid)
    to OT maps via quantile computation.
    
    For each spectrum (a PMF on n_bands uniform grid points), compute
    the inverse-CDF evaluated on a uniform grid — i.e., the OT map
    from Uniform[0,1] to the spectrum.
    
    Args:
        spectra: (N, n_bands) array of normalized spectra (rows sum to 1)
        n_bands: number of spectral bands
    
    Returns:
        maps: (N, n_bands) array of OT maps
    """
    spectra = np.asarray(spectra, dtype=np.float64)
    N, D = spectra.shape
    
    # Grid where the spectral bands live
    x_grid = np.linspace(0.0, 1.0, D)
    # Uniform quantile grid to evaluate the inverse-CDF on
    p_grid = np.linspace(0.0, 1.0, D)
    
    maps = np.zeros_like(spectra)
    for i in range(N):
        pmf = spectra[i]
        # CDF from the PMF
        cdf = np.cumsum(pmf)
        cdf = cdf / cdf[-1]  # ensure ends at 1
        # Inverse-CDF: for each p in p_grid, find the smallest x with CDF(x) >= p
        maps[i] = np.interp(p_grid, cdf, x_grid)
    
    return maps

In [ ]:
# ── SAE model loading and reconstruction ──

def model_path_for(key):
    return SAE_ROOT / key / 'sparse_ae.pt'


def load_sae_model(architecture, hidden_dim, model_path, top_k=None):
    """Load a trained SAE model."""
    cls = ARCH_REGISTRY[architecture]
    if cls in (TopKAE, TopKAE_monotone):
        model = cls(N_BANDS, hidden_dim, top_k=top_k)
    else:
        model = cls(N_BANDS, hidden_dim)
    model = model.to(DEVICE)
    state = torch.load(model_path, map_location=DEVICE)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    model.load_state_dict(state, strict=False)
    model.eval()
    return model


def sae_reconstruct(key, cfg, data):
    """
    Encode data through SAE, then reconstruct via z @ A.
    Returns (reconstructed_maps, codes).
    """
    # Encode
    codes = sae_encode_potential_batch(
        data, model_path_for(key), cfg['architecture'],
        N_BANDS, cfg['hidden_dim'], cfg['top_K'],
        device=DEVICE, batch_size=1024
    )
    
    # Reconstruct: z @ A (with start-at-zero correction)
    model = load_sae_model(cfg['architecture'], cfg['hidden_dim'],
                           model_path_for(key), top_k=cfg.get('top_K'))
    with torch.no_grad():
        A = model.atoms()
        xhat = codes.to(A.device, dtype=A.dtype) @ A
        xhat = xhat - xhat[..., :1]
    
    return xhat.cpu().numpy(), codes.numpy()

## 3. Run SAE Reconstructions (in OT map space)

In [ ]:
sae_recon_maps = {}   # key → (N, D) reconstructed OT maps
sae_codes = {}        # key → (N, hidden_dim) codes

for key, cfg in SAE_CONFIGS.items():
    print(f'SAE: {key} ...', end=' ')
    recon, codes = sae_reconstruct(key, cfg, transport_map_data)
    sae_recon_maps[key] = recon
    sae_codes[key] = codes
    mse = np.mean((transport_map_data.numpy() - recon) ** 2)
    print(f'map MSE = {mse:.6e}')

## 4. Run NMF Reconstructions (in spectral space)

In [ ]:
nmf_models = {}       # rank → fitted NMF model
nmf_W = {}            # rank → (N, rank) coefficients
nmf_H = {}            # rank → (rank, D) dictionary
nmf_recon_spectra = {} # rank → (N, D) reconstructed normalized spectra

for rank in NMF_RANKS:
    print(f'NMF rank={rank} ...', end=' ')
    model = NMF(n_components=rank, init='nndsvda', max_iter=1200, random_state=42)
    W = model.fit_transform(spectra_normalized)
    H = model.components_
    recon = W @ H
    
    nmf_models[rank] = model
    nmf_W[rank] = W
    nmf_H[rank] = H
    nmf_recon_spectra[rank] = recon
    
    mse = np.mean((spectra_normalized - recon) ** 2)
    print(f'spectral MSE = {mse:.6e}, iters = {model.n_iter_}')

## 5. Comparison A — L2 Error on Spectral Densities

For the SAE method, we need to convert reconstructed OT maps → spectral densities via KDE.  
First, tune the KDE hyperparameters to find the best settings.

In [ ]:
# ── KDE hyperparameter grid search ──
# We tune on a random subset for speed, then apply the best to all pixels.

TUNE_N = 500          # number of pixels to tune on
BANDWIDTH_GRID = [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.1]
N_SAMPLES_GRID = [5000, 20000, 50000]

np.random.seed(42)
tune_idx = np.random.choice(N, size=min(TUNE_N, N), replace=False)

# Pick one SAE model to tune on (the first one)
tune_key = list(SAE_CONFIGS.keys())[0]
tune_maps = sae_recon_maps[tune_key][tune_idx]
tune_spectra = spectra_normalized[tune_idx]

print(f'Tuning KDE on {len(tune_idx)} pixels using SAE model: {tune_key}')
print(f'Bandwidth grid: {BANDWIDTH_GRID}')
print(f'N_samples grid: {N_SAMPLES_GRID}')
print()

tune_results = []
for bw, ns in product(BANDWIDTH_GRID, N_SAMPLES_GRID):
    errors = []
    for i in range(len(tune_idx)):
        _, density = kde_density_from_inv_cdf(tune_maps[i], n_samples=ns,
                                              bandwidth=bw, n_grid=D)
        err = np.mean((tune_spectra[i] - density) ** 2)
        errors.append(err)
    mean_mse = np.mean(errors)
    tune_results.append({'bandwidth': bw, 'n_samples': ns, 'mse': mean_mse})
    print(f'  bw={bw:<6.3f}  n_samples={ns:<6d}  MSE={mean_mse:.6e}')

tune_results.sort(key=lambda x: x['mse'])
best = tune_results[0]
BEST_BW = best['bandwidth']
BEST_NS = best['n_samples']
print(f'\nBest: bandwidth={BEST_BW}, n_samples={BEST_NS}, MSE={best["mse"]:.6e}')

In [ ]:
# ── Convert all SAE reconstructions to spectra using best KDE params ──

sae_recon_spectra = {}

for key in SAE_CONFIGS:
    print(f'Converting SAE maps → spectra for {key} ...')
    maps = sae_recon_maps[key]
    spectra_out = np.zeros((N, D))
    for i in range(N):
        _, spectra_out[i] = kde_density_from_inv_cdf(
            maps[i], n_samples=BEST_NS, bandwidth=BEST_BW, n_grid=D
        )
    sae_recon_spectra[key] = spectra_out

In [ ]:
# ── Spectral L2 error comparison ──

print(f'{"Method":<40s}  {"Mean L2 (spectral)":>18s}')
print('=' * 62)

spectral_results = {}

for key in SAE_CONFIGS:
    per_pixel = np.mean((spectra_normalized - sae_recon_spectra[key]) ** 2, axis=1)
    mean_l2 = per_pixel.mean()
    spectral_results[f'SAE: {key}'] = mean_l2
    print(f'SAE: {key:<36s}  {mean_l2:18.6e}')

for rank in NMF_RANKS:
    per_pixel = np.mean((spectra_normalized - nmf_recon_spectra[rank]) ** 2, axis=1)
    mean_l2 = per_pixel.mean()
    spectral_results[f'NMF: rank={rank}'] = mean_l2
    print(f'NMF: rank={rank:<33d}  {mean_l2:18.6e}')

## 6. Comparison B — L2 Error on Transport Maps

- **SAE**: already operates in OT map space, so error = `||T_original - T_reconstructed||^2`
- **NMF**: reconstruct spectra, then convert to OT maps via quantile computation

In [ ]:
# Ground-truth OT maps
gt_maps = transport_map_data.numpy() if torch.is_tensor(transport_map_data) else np.asarray(transport_map_data)

print(f'{"Method":<40s}  {"Mean L2 (map)":>18s}')
print('=' * 62)

map_results = {}

# SAE: direct comparison in map space
for key in SAE_CONFIGS:
    per_pixel = np.mean((gt_maps - sae_recon_maps[key]) ** 2, axis=1)
    mean_l2 = per_pixel.mean()
    map_results[f'SAE: {key}'] = mean_l2
    print(f'SAE: {key:<36s}  {mean_l2:18.6e}')

# NMF: reconstruct spectra → compute OT map → compare
for rank in NMF_RANKS:
    print(f'NMF rank={rank}: converting reconstructed spectra → OT maps ...')
    # Clip and renormalize NMF reconstruction to valid PMF
    recon = np.clip(nmf_recon_spectra[rank], 0, None)
    sums = recon.sum(axis=1, keepdims=True)
    sums = np.where(sums < 1e-12, 1.0, sums)
    recon_pmf = recon / sums
    
    nmf_maps = spectra_to_ot_maps(recon_pmf, D)
    per_pixel = np.mean((gt_maps - nmf_maps) ** 2, axis=1)
    mean_l2 = per_pixel.mean()
    map_results[f'NMF: rank={rank}'] = mean_l2
    print(f'NMF: rank={rank:<33d}  {mean_l2:18.6e}')

## 7. Summary

In [ ]:
print(f'Dataset: {DATASET_NAME}  |  N = {N}  |  D = {D} bands')
print(f'Best KDE params: bandwidth={BEST_BW}, n_samples={BEST_NS}')
print()

# Collect all methods
all_methods = list(spectral_results.keys())

print(f'{"Method":<40s}  {"Spectral L2":>14s}  {"Map L2":>14s}')
print('=' * 72)
for m in all_methods:
    s = spectral_results.get(m, float('nan'))
    t = map_results.get(m, float('nan'))
    print(f'{m:<40s}  {s:14.6e}  {t:14.6e}')

In [ ]:
# ── Bar chart comparison ──

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = list(spectral_results.keys())
colors = ['#1f77b4' if 'SAE' in m else '#d62728' for m in methods]
short_names = [m.replace('SAE: ', '').replace('NMF: ', 'NMF ') for m in methods]

# Spectral L2
vals = [spectral_results[m] for m in methods]
axes[0].barh(range(len(methods)), vals, color=colors)
axes[0].set_yticks(range(len(methods)))
axes[0].set_yticklabels(short_names, fontsize=8)
axes[0].set_xlabel('Mean L2 Error')
axes[0].set_title('Spectral Density Reconstruction')
axes[0].invert_yaxis()

# Map L2
vals = [map_results[m] for m in methods]
axes[1].barh(range(len(methods)), vals, color=colors)
axes[1].set_yticks(range(len(methods)))
axes[1].set_yticklabels(short_names, fontsize=8)
axes[1].set_xlabel('Mean L2 Error')
axes[1].set_title('Transport Map Reconstruction')
axes[1].invert_yaxis()

fig.suptitle(f'{DATASET_NAME} — Reconstruction Error Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Example pixel comparisons ──

np.random.seed(0)
example_idx = np.random.choice(N, size=5, replace=False)

example_sae_key = list(SAE_CONFIGS.keys())[0]
example_nmf_rank = NMF_RANKS[0]

for idx in example_idx:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    x = np.linspace(0, 1, D)
    
    # Panel 1: Transport maps
    axes[0].plot(x, gt_maps[idx], label='Original', lw=1.2)
    axes[0].plot(x, sae_recon_maps[example_sae_key][idx], '--', label=f'SAE ({example_sae_key})', lw=1.2)
    
    # NMF map for this pixel
    recon_pmf = np.clip(nmf_recon_spectra[example_nmf_rank][idx], 0, None)
    recon_pmf = recon_pmf / (recon_pmf.sum() + 1e-12)
    nmf_map = spectra_to_ot_maps(recon_pmf.reshape(1, -1), D)[0]
    axes[0].plot(x, nmf_map, ':', label=f'NMF (rank={example_nmf_rank})', lw=1.2)
    
    axes[0].set(xlabel='Quantile $p$', ylabel='$T(p)$', title=f'Transport Maps — pixel {idx}')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    # Panel 2: Spectra
    axes[1].plot(x, spectra_normalized[idx], label='Original', lw=1.2)
    axes[1].plot(x, sae_recon_spectra[example_sae_key][idx], '--',
                 label=f'SAE ({example_sae_key})', lw=1.2)
    axes[1].plot(x, nmf_recon_spectra[example_nmf_rank][idx], ':',
                 label=f'NMF (rank={example_nmf_rank})', lw=1.2)
    axes[1].set(xlabel='Band', ylabel='Normalized intensity', title=f'Spectra — pixel {idx}')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()